# Indian Primate Species Classifier
**Source:** iNaturalist API | **Model:** EfficientNetB3 Transfer Learning

Species: Rhesus Macaque, Bonnet Macaque, Lion-tailed Macaque, Hanuman Langur, Golden Langur, Hoolock Gibbon

> Before running: Settings → Accelerator → GPU P100 | Settings → Internet → ON

In [ ]:
# CELL 1 — Imports
import os, io, json, shutil, random, requests, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('Ready')

In [ ]:
# CELL 2 — Config
# iNaturalist taxon IDs (verified from inaturalist.org)
SPECIES = {
    'rhesus_macaque':       43460,
    'bonnet_macaque':       43448,
    'lion_tailed_macaque':  43446,
    'hanuman_langur':       1071032,
    'golden_langur':        43495,
    'hoolock_gibbon':       74439,
}
SPECIES_LIST    = list(SPECIES.keys())
PLACE_ID        = 6681     # India
MAX_PER_SPECIES = 150      # images per species
IMG_SIZE        = 300      # EfficientNetB3 input
BATCH_SIZE      = 16
CONFIDENCE_THR  = 0.70

DATASET_DIR = '/kaggle/working/dataset'
SPLIT_DIR   = '/kaggle/working/split'
MODEL_DIR   = '/kaggle/working/models'
for d in [DATASET_DIR, SPLIT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config ready')
print(f'Species: {SPECIES_LIST}')
print(f'Max images per species: {MAX_PER_SPECIES}')
print(f'Total target images: {MAX_PER_SPECIES * len(SPECIES_LIST)}')

In [ ]:
# CELL 3 — Download images from iNaturalist API
INAT_API = 'https://api.inaturalist.org/v1/observations'

def get_photo_urls(taxon_id, place_id, max_images):
    urls, page = [], 1
    while len(urls) < max_images:
        try:
            r = requests.get(INAT_API, params={
                'taxon_id':      taxon_id,
                'place_id':      place_id,
                'quality_grade': 'research',
                'photos':        'true',
                'per_page':      100,
                'page':          page,
                'order':         'desc',
                'order_by':      'created_at'
            }, timeout=15)
            results = r.json().get('results', [])
            if not results:
                break
            for obs in results:
                for photo in obs.get('photos', []):
                    url = photo.get('url', '').replace('square', 'medium')
                    if url:
                        urls.append(url)
                        if len(urls) >= max_images:
                            break
                if len(urls) >= max_images:
                    break
            page += 1
            time.sleep(0.5)
        except Exception as e:
            print(f'  API error: {e}')
            break
    return urls

def download_one(args):
    url, path = args
    if os.path.exists(path):
        return True
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            Image.open(io.BytesIO(r.content)).verify()
            open(path, 'wb').write(r.content)
            return True
    except:
        pass
    return False

print('Fetching URLs and downloading images...\n')
all_tasks = []

for name, taxon_id in SPECIES.items():
    folder = os.path.join(DATASET_DIR, name)
    os.makedirs(folder, exist_ok=True)
    print(f'  Fetching URLs: {name}...')
    urls = get_photo_urls(taxon_id, PLACE_ID, MAX_PER_SPECIES)
    print(f'  Found {len(urls)} URLs')
    for i, url in enumerate(urls):
        ext = url.split('.')[-1].split('?')[0]
        if ext not in ['jpg','jpeg','png']:
            ext = 'jpg'
        all_tasks.append((url, os.path.join(folder, f'{i}.{ext}')))

print(f'\nDownloading {len(all_tasks)} images (8 workers)...')
ok = fail = 0
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(download_one, t) for t in all_tasks]
    for i, f in enumerate(as_completed(futures)):
        if f.result(): ok += 1
        else: fail += 1
        if (i+1) % 100 == 0:
            print(f'  {i+1}/{len(all_tasks)} | OK={ok} FAIL={fail}')

print(f'\nDone | Downloaded={ok} | Failed={fail}')
print('\nImages per species:')
for name in SPECIES_LIST:
    n = len(os.listdir(os.path.join(DATASET_DIR, name)))
    print(f'  {name:<25} {n}')

# Check disk space
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'\nDisk free: {free/1024**3:.1f} GB')

In [ ]:
# CELL 4 — Train/Val/Test Split (70/15/15)
exts = {'.jpg','.jpeg','.png','.webp'}

if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

for split in ['train','val','test']:
    for name in SPECIES_LIST:
        os.makedirs(os.path.join(SPLIT_DIR, split, name), exist_ok=True)

print('Splitting dataset 70/15/15...')
for name in SPECIES_LIST:
    imgs = [f for f in os.listdir(os.path.join(DATASET_DIR, name))
            if Path(f).suffix.lower() in exts]
    random.shuffle(imgs)
    n = len(imgs)
    n_tr = int(n * 0.70)
    n_va = int(n * 0.15)
    splits = {'train': imgs[:n_tr],
              'val':   imgs[n_tr:n_tr+n_va],
              'test':  imgs[n_tr+n_va:]}
    for split, files in splits.items():
        for f in files:
            shutil.copy2(
                os.path.join(DATASET_DIR, name, f),
                os.path.join(SPLIT_DIR, split, name, f)
            )
    print(f'  {name:<25} train={len(splits["train"])} val={len(splits["val"])} test={len(splits["test"])}')
print('Split done')

In [ ]:
# CELL 5 — Data Generators + Class Weights
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.25,
    height_shift_range=0.25,
    shear_range=0.2,
    zoom_range=0.25,
    horizontal_flip=True,
    brightness_range=[0.6, 1.4],
    channel_shift_range=25.0,
    fill_mode='nearest'
).flow_from_directory(
    os.path.join(SPLIT_DIR,'train'),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=SPECIES_LIST,
    shuffle=True
)

eval_dg = ImageDataGenerator(rescale=1./255)
val_gen = eval_dg.flow_from_directory(
    os.path.join(SPLIT_DIR,'val'),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=SPECIES_LIST,
    shuffle=False
)
test_gen = eval_dg.flow_from_directory(
    os.path.join(SPLIT_DIR,'test'),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=SPECIES_LIST,
    shuffle=False
)

# Class weights to handle imbalance
cw_vals = compute_class_weight('balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes)
class_weights = dict(enumerate(cw_vals))

idx2cls = {v:k for k,v in train_gen.class_indices.items()}
print('Class weights:')
for i,w in class_weights.items():
    print(f'  {idx2cls[i]:<25} {w:.3f}')

In [ ]:
# CELL 6 — Build EfficientNetB3 Model
base = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(len(SPECIES_LIST), activation='softmax')(x)

model = Model(base.input, out)
model.compile(
    optimizer=Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f'Total params:     {model.count_params():,}')
print(f'Trainable params: {sum(np.prod(v.shape) for v in model.trainable_variables):,} (head only)')
print('Model ready')

In [ ]:
# CELL 7 — Phase 1: Train head only (base frozen)
print('PHASE 1 — Training head only')
h1 = model.fit(
    train_gen, epochs=15,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=5,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(os.path.join(MODEL_DIR,'phase1.keras'),
                        monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1)
    ], verbose=1
)
print(f'Phase 1 done | Best val acc: {max(h1.history["val_accuracy"]):.4f}')

In [ ]:
# CELL 8 — Phase 2: Fine-tune top 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f'Trainable params now: {sum(np.prod(v.shape) for v in model.trainable_variables):,}')
print('PHASE 2 — Fine-tuning top 30 layers')

h2 = model.fit(
    train_gen, epochs=25,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=7,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(os.path.join(MODEL_DIR,'final_model.keras'),
                        monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                          patience=4, min_lr=1e-8, verbose=1)
    ], verbose=1
)
print(f'Phase 2 done | Best val acc: {max(h2.history["val_accuracy"]):.4f}')

In [ ]:
# CELL 9 — Evaluate + Confusion Matrix
test_gen.reset()
preds       = model.predict(test_gen, verbose=1)
true_labels = test_gen.classes
pred_labels = np.argmax(preds, axis=1)
max_confs   = np.max(preds, axis=1)

acc     = np.mean(pred_labels == true_labels)
mf1     = f1_score(true_labels, pred_labels, average='macro')
top2    = np.argsort(preds, axis=1)[:, -2:]
top2acc = np.mean([true_labels[i] in top2[i] for i in range(len(true_labels))])
uncert  = np.sum(max_confs < CONFIDENCE_THR)

print(f'Test Accuracy   : {acc*100:.1f}%')
print(f'Mean F1 (macro) : {mf1:.3f}')
print(f'Top-2 Accuracy  : {top2acc*100:.1f}%')
print(f'Uncertain (<{CONFIDENCE_THR}): {uncert}/{len(true_labels)} ({100*uncert/len(true_labels):.1f}%)')
print()
print(classification_report(true_labels, pred_labels,
                             target_names=SPECIES_LIST, digits=3))

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[s.replace('_','\n') for s in SPECIES_LIST],
            yticklabels=[s.replace('_','\n') for s in SPECIES_LIST])
plt.title('Confusion Matrix — Indian Primate Classifier', fontweight='bold')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR,'confusion_matrix.png'), dpi=150)
plt.show()

# Save metrics
metrics = {
    'accuracy': float(acc),
    'mean_f1': float(mf1),
    'top2_accuracy': float(top2acc),
    'pct_uncertain': float(100*uncert/len(true_labels))
}

In [ ]:
# CELL 10 — Save Model + Print Resume Metrics
model.save(os.path.join(MODEL_DIR, 'primate_classifier.h5'))
json.dump({
    'species': SPECIES_LIST,
    'class_indices': train_gen.class_indices,
    'img_size': IMG_SIZE,
    'confidence_threshold': CONFIDENCE_THR,
    'metrics': metrics,
    'trained_at': datetime.now().isoformat()
}, open(os.path.join(MODEL_DIR,'model_info.json'),'w'), indent=2)

total_imgs = sum(len(os.listdir(os.path.join(DATASET_DIR,s))) for s in SPECIES_LIST)

print('='*60)
print('RESUME METRICS')
print('='*60)
print(f"""
  Architecture : EfficientNetB3 + Two-phase Transfer Learning
  Dataset      : {total_imgs} research-grade images from iNaturalist
                 India only, {len(SPECIES_LIST)} primate species

  Test Accuracy  : {metrics['accuracy']*100:.1f}%
  Mean F1 Score  : {metrics['mean_f1']:.3f}
  Top-2 Accuracy : {metrics['top2_accuracy']*100:.1f}%

  RESUME LINE:
  Built a fine-grained Indian primate species classifier
  using EfficientNetB3 transfer learning with two-phase
  fine-tuning — achieving {metrics['accuracy']*100:.0f}% accuracy and
  {metrics['mean_f1']:.2f} mean F1 across {len(SPECIES_LIST)} visually similar species.
  Dataset curated from iNaturalist Open Data API
  filtering research-grade India observations.

  Saved: /kaggle/working/models/
    primate_classifier.h5
    model_info.json
    confusion_matrix.png
""")
print('='*60)